In [8]:
import pandas as pd 
import numpy as np 
import re
from tableone import TableOne

pd.set_option('display.max_rows', 100)  # Show up to 100 rows
pd.set_option('display.max_columns', None)

In [9]:
path = '/Users/herbert.kapesa/Library/CloudStorage/OneDrive-CentreforInfectiousDiseaseResearchinZambia/RMNCH/PPD 2/Data extract/Latest Extract/KMC_PPD_20260805.csv'
df = pd.read_csv(path)

/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_47109/2258068660.py:2: DtypeWarning: Columns (8,25,26,27,28,29,30,31,32,33,34,79,80,85,86,87,88,89,90,92,93,94,95,96,97,98,117,130,131,132,133,135,137,169,180,181,182,189,190,191,192,194,199,200,203,204,206,209,210,212,214,216,218,220,224,225,239,243,245,279,280,284,285,289,290,291,295,303,304,306,308,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,335,336,337,338,339,340,341,342,343,344,345,347,349,350,351,353,354,355,356,357,359,372,376,378,412,416,417,418,422,423,424,427,429,430,439,440,442,444,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,472,473,474,475,476,477,478,479,480,481,482,484,486,487,488,489,491,492,493,494,496,498,499,501,502,503,504,505,506,507,508,509,510,512,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,533,534,535,536,537,538,539,540,541,542,543,545,547,548,549,552,553,554,555,557,558,559,560,56

In [10]:
df.columns

Index(['participant_id', 'redcap_event_name', 'redcap_repeat_instrument',
       'redcap_repeat_instance', 'notes', 'participant_chart_notes_complete',
       'vi_m_schedule', 'vi_m_motherdate', 'vi_f_fatherdate',
       'vi_f_fatherdate_na',
       ...
       'ci_f_name', 'ci_f_aka', 'ci_f_phone1', 'ci_f_phone2', 'ci_f_name_sc',
       'ci_f_phone_sc1', 'ci_f_phone_sc2', 'ci_f_adress', 'ci_f_map',
       'contact_information_father_complete'],
      dtype='object', length=1041)

In [11]:
df["participant_id"]

0       PPD-M-001
1       PPD-M-001
2       PPD-M-001
3       PPD-M-001
4       PPD-M-001
          ...    
7531    PPD-M-612
7532    PPD-M-612
7533    PPD-M-612
7534    PPD-M-612
7535    PPD-M-612
Name: participant_id, Length: 7536, dtype: object

In [12]:
## Combine participant crfs into one record

def _combine_series(s):
    vals = s.dropna().astype(str).unique()
    if vals.size == 0:
        return np.nan
    if vals.size == 1:
        v = vals[0]
        if pd.api.types.is_numeric_dtype(s):
            try:
                return pd.to_numeric(v)
            except:
                return v
        if pd.api.types.is_bool_dtype(s):
            return v.lower() in ("true", "1", "yes")
        return v
    return " | ".join(vals)

dedup_df = df.groupby("participant_id", sort=False, as_index=False).agg(_combine_series)
print(f"rows before: {len(df)}, rows after: {len(dedup_df)}")
df = dedup_df.copy()

rows before: 7536, rows after: 612


/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_47109/2991340239.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dedup_df = df.groupby("participant_id", sort=False, as_index=False).agg(_combine_series)


In [13]:
df[['participant_id',"en_m_dob"]].loc[df['en_m_dob'].isna()]

,participant_id,en_m_dob
564,PPD-M-565,NaN


In [14]:
df[['participant_id',"en_m_dob"]].loc[df['en_m_dob'].isna()]

,participant_id,en_m_dob
564,PPD-M-565,NaN


In [15]:
# Convert date strings to datetime
df["en_date"] = pd.to_datetime(df["en_date"], errors="coerce")
df["en_m_dob"] = pd.to_datetime(df["en_m_dob"], errors="coerce")

# Calculate age at enrollment in completed years (no decimal places)
df["mothers_age"] = (
    np.floor((df["en_date"] - df["en_m_dob"]).dt.days / 365.25)
    .astype("Int64")
)
df['mothers_age_group'] = pd.cut(df['mothers_age'], 
                                  bins=[14, 19, 24, 29, 34, 39, 100],
                                  labels=['15-19 years', '20-24 years', '25-29 years', 
                                          '30-34 years', '35-39 years', '40+ years'])

/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_47109/3991350160.py:2: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["en_date"] = pd.to_datetime(df["en_date"], errors="coerce")
/var/folders/r2/83tq991d3ng89thh8jc6hk51090g_x/T/ipykernel_47109/3991350160.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df["en_m_dob"] = pd.to_datetime(df["en_m_dob"], errors="coerce")


In [16]:
# work on parity and gravida
df["en_para"] = pd.to_numeric(df["en_para"], errors="coerce")
df['mothers_par_group'] = pd.cut(df['en_para'], 
                                  bins=[-1,0,1,2, 100],
                                  labels=['0','1', '2', '3+'])

# Work on gravida
df["en_gra"] = pd.to_numeric(df["en_gra"], errors="coerce")
df['mothers_grav_group'] = pd.cut(df['en_gra'], 
                                  bins=[-1,0,1,2, 100],
                                  labels=['0','1', '2', '3+'])          


# Sexual debut age categorization
df["en_m_asxd"] = pd.to_numeric(df["en_m_asxd"], errors="coerce")
df['mothers_sex_debut_group'] = pd.cut(df['en_m_asxd'],
                                       bins=[-1, 14, 18, 100],
                                       labels=['<15 years', '15-18 years', ">18 years"])
# Number of ANC visits categorization
df["en_m_anc"] = pd.to_numeric(df["en_m_anc"], errors="coerce")
df['mothers_anc_group'] = pd.cut(df['en_m_anc'],
                                  bins=[-1, 0,1,2,3, 4, 100],
                                  labels=['0','1','2','3','4', '5+'])
# EPDS score categorization
df["fp_m_scr_epds_sco"] = pd.to_numeric(df["fp_m_scr_epds_sco"], errors="coerce")
df['mothers_epds_group_2weeks'] = pd.cut(df['fp_m_scr_epds_sco'],
                                  bins=[-1,10, 100],
                                  labels=[ 'Negative', 'Positive'])

# EPDS score categorization 6-8 weeks
df["fp1_m_scr_epds_sco"] = pd.to_numeric(df["fp1_m_scr_epds_sco"], errors="coerce")
df['mothers_epds_group_6-8weeks'] = pd.cut(df['fp1_m_scr_epds_sco'],
                                  bins=[-1,10, 100],
                                  labels=[ 'Negative', 'Positive'])     
df[['mothers_epds_group_6-8weeks','fp1_m_scr_epds_sco']]


,mothers_epds_group_6-8weeks,fp1_m_scr_epds_sco
0,Negative,6.0
1,Negative,8.0
2,Negative,6.0
3,Negative,8.0
4,Negative,8.0
...,...,...
607,Negative,0.0
608,Negative,0.0
609,Negative,7.0
610,Negative,9.0


In [17]:
df[['participant_id',"mothers_age_group","mothers_age","en_m_dob"]].loc[df['mothers_age_group']=="15-19 years"]

,participant_id,mothers_age_group,mothers_age,en_m_dob
1,PPD-M-002,15-19 years,18,2005-12-09
9,PPD-M-010,15-19 years,18,2006-05-21
12,PPD-M-013,15-19 years,19,2005-05-23
42,PPD-M-042,15-19 years,19,2006-01-01
65,PPD-M-065,15-19 years,18,2007-02-01
89,PPD-M-089,15-19 years,19,2005-04-14
90,PPD-M-090,15-19 years,19,2005-08-15
91,PPD-M-091,15-19 years,19,2005-11-09
94,PPD-M-094,15-19 years,19,2005-11-22
106,PPD-M-106,15-19 years,18,2007-02-23


In [23]:
df["en_m_mstat"].value_counts()

en_m_mstat
Married              469
In a relationship    119
Single                11
Separated             10
Widowed                1
Divorced               1
None                   1
Name: count, dtype: int64

In [24]:
columns = [ "mothers_age_group",
"mothers_par_group",
"mothers_grav_group",
"en_misc",
"en_mod",
"en_m_hgt",
"en_m_wgt",
"mothers_sex_debut_group",
"en_m_mstat",
"en_m_hiv",
"en_m_sku",
"en_m_inc",
"en_m_mlev",
"mothers_anc_group",
"en_m_smk",
"en_m_drg",
"en_m_alc",
"mothers_epds_group_2weeks",
'mothers_epds_group_6-8weeks'
]
categorical = ['mothers_age_group',
"mothers_par_group",
"mothers_grav_group",
"en_misc",
"en_mod",
"mothers_sex_debut_group",
"en_m_mstat",
"en_m_hiv",
"en_m_sku",
"en_m_inc",
"en_m_mlev",
"mothers_anc_group",
"en_m_smk",
"en_m_drg",
"en_m_alc",
"mothers_epds_group_2weeks",
'mothers_epds_group_6-8weeks'
]

continuous = ["en_m_hgt","en_m_wgt"]
min_max = ["en_m_hgt","en_m_wgt"]
nonnormal = ["mothers_par_group","mothers_grav_group"]



lables = {
 "mothers_age_group": "Mother's Age Group",
"mothers_par_group": "Mother's Parity Group",
"mothers_grav_group": "Mother's Gravida Group",
"en_misc": "Miscarriages",
"en_mod": "Mode of Delivery",
"en_m_hgt": "Mother's Height",
"en_m_wgt": "Mother's Weight",
"mothers_sex_debut_group": "Mother's Sexual Debut Age Group",
"en_m_mstat": "Mother's Marital Status",
"en_m_hiv": "Mother's HIV Status",
"en_m_sku": "Mother's Education Status",
"en_m_inc": "Mother's Source of Income ",
"en_m_mlev": "Maternal leave",
"mothers_anc_group": "Mother's ANC Visits",
"en_m_smk": "Mother's Smoking Status",
"en_m_drg": "Mother's Drug Use Status",
"en_m_alc": "Mother's Alcohol Use Status",
"mothers_epds_group_2weeks": "Mother's EPDS Score (2 weeks)",
"mothers_epds_group_6-8weeks": "Mother's EPDS Score (6-8 weeks)"
}   

groupby = 'en_rand'

In [25]:
mytable = TableOne(df, columns, categorical, continuous, groupby,min_max=min_max,rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)

In [26]:


mytable

Grouped by en_rand                                                                                                                                       
                                                                                        Missing              Overall Control (NICU → KMC ward for < 48 hours → home) Intervention (Immediate admission to the KMC Ward → home) P-Value
n                                                                                                                612                                             306                                                       306        
Mother's Age Group, n (%)              15-19 years                                                         84 (13.7)                                       45 (14.7)                                                 39 (12.7)   0.279
                                       20-24 years                                                        161 (26.3)                                       79 (25.8)                                                 82 (26.8)        
                                       25-29 years                                                        160 (26.1)                                       84 (27.5)                                                 76 (24.8)        
                                       30-34 years                                                        124 (20.3)                                       66 (21.6)                                                 58 (19.0)        
                                       35-39 years                                                          58 (9.5)                                        21 (6.9)                                                 37 (12.1)        
                                       40+ years                                                            24 (3.9)                                        10 (3.3)                                                  14 (4.6)        
                                       None                                                                  1 (0.2)                                         1 (0.3)                                                   0 (0.0)        
Mother's Parity Group, n (%)           1                                                                  192 (31.4)                                      100 (32.7)                                                 92 (30.1)   0.201
                                       2                                                                  140 (22.9)                                       77 (25.2)                                                 63 (20.6)        
                                       3+                                                                 279 (45.6)                                      128 (41.8)                                                151 (49.3)        
                                       None                                                                  1 (0.2)                                         1 (0.3)                                                   0 (0.0)        
Mother's Gravida Group, n (%)          1                                                                  175 (28.6)                                       91 (29.7)                                                 84 (27.5)   0.172
                                       2                                                                  150 (24.5)                                       83 (27.1)                                                 67 (21.9)        
                                       3+                                                                 286 (46.7)                                      131 (42.8)                                                155 (50.7)        
                                       None                                                                  1 (0.2)                                  

In [ ]:
df['mothers_epds_group_2weeks'].value_counts()

mothers_epds_group_2weeks
Negative    440
Positive     84
None          6
Name: count, dtype: int64

In [ ]:
df = df.loc[df['mothers_epds_group_2weeks'] != 'None']

In [ ]:
epds_table = TableOne(df, columns, categorical, continuous, groupby = "mothers_epds_group_2weeks",min_max=min_max,rename=lables,label_suffix=True, normal_test=True, tukey_test=True, pval=True)

In [ ]:
epds_table

Grouped by Mother's EPDS Score (2 weeks)                                                                       
                                                                                                             Missing              Overall             Negative             Positive P-Value
n                                                                                                                                     524                  440                   84        
Mother's Age Group, n (%)              15-19 years                                                                              70 (13.4)            60 (13.6)            10 (11.9)   0.466
                                       20-24 years                                                                             136 (26.0)           118 (26.8)            18 (21.4)        
                                       25-29 years                                                                             137 (26.1)           111 (25.2)            26 (31.0)        
                                       30-34 years                                                                             109 (20.8)            91 (20.7)            18 (21.4)        
                                       35-39 years                                                                               51 (9.7)            45 (10.2)              6 (7.1)        
                                       40+ years                                                                                 20 (3.8)             14 (3.2)              6 (7.1)        
                                       None                                                                                       1 (0.2)              1 (0.2)              0 (0.0)        
Mother's Parity Group, n (%)           1                                                                                       172 (32.8)           150 (34.1)            22 (26.2)   0.032
                                       2                                                                                       121 (23.1)           108 (24.5)            13 (15.5)        
                                       3+                                                                                      230 (43.9)           181 (41.1)            49 (58.3)        
                                       None                                                                                       1 (0.2)              1 (0.2)              0 (0.0)        
Mother's Gravida Group, n (%)          1                                                                                       154 (29.4)           134 (30.5)            20 (23.8)   0.034
                                       2                                                                                       132 (25.2)           118 (26.8)            14 (16.7)        
                                       3+                                                                                      237 (45.2)           187 (42.5)            50 (59.5)        
                                       None                                                                                       1 (0.2)              1 (0.2)              0 (0.0)        
Mother's Marital Status, n (%)         0.0                                                                                     418 (79.8)           361 (82.0)            57 (67.9)   0.039
                                       1.0                                                                                      77 (14.7)            55 (12.5)            22 (26.2)        
                                       2.0                                                                                       17 (3.2)             13 (3.0)              4 (4.8)        
                                       3.0                                                                                      

In [40]:
df['mothers_epds_group_2weeks'].value_counts()

mothers_epds_group_2weeks
Negative    500
Positive     84
None         28
Name: count, dtype: int64

In [ ]:
# Create an Excel writer object
with pd.ExcelWriter('/Users/herbert.kapesa/Library/CloudStorage/OneDrive-CentreforInfectiousDiseaseResearchinZambia/RMNCH/PPD 2/output_tables/ppd_tables.xlsx', engine='openpyxl') as writer:
    # Write the first table to the first sheet
    mytable.tableone.to_excel(writer, sheet_name='Overall Table')
    
    # Write the second table to the second sheet
    epds_table.tableone.to_excel(writer, sheet_name='EPDS Table')

In [101]:
df['2weeks'] = df['mothers_epds_group_2weeks']
df['6_8weeks'] = df['mothers_epds_group_6-8weeks']

# Copy database for logistic regression analysis
df_logit = df[['participant_id','2weeks','6_8weeks','en_rand']].copy()

# data variable transformation for logistic regression
# Randomisation group
df_logit['en_rand'] = df_logit['en_rand'].map({'Control (NICU → KMC ward for < 48 hours → home)': 'Control', 'Intervention (Immediate admission to the KMC Ward → home)': 'Intervention'})
df_logit['en_rand_num'] = df_logit['en_rand'].map({ 'Control' : 0, 'Intervention' : 1 })
# Timepoint



In [102]:
# tranform dataframe from wide to long format for logistic regression analysis

df_logit_long = pd.melt(
    df_logit, 
    id_vars=['participant_id','en_rand'], 
    value_vars=['2weeks', '6_8weeks'], 
    var_name='timepoint', value_name='epds_result')

df_logit_long.sort_values(by=['participant_id'], inplace=True)

# Change the 'timepoint' and 'epds_result' columns to numeric values for logistic regression
df_logit_long['timepoint_num'] = df_logit_long['timepoint'].map({'2weeks': 0, '6_8weeks': 1})
df_logit_long['epds_result_num'] = df_logit_long['epds_result'].map({'Negative': 0, 'Positive': 1})


In [103]:
# check for duplicate ids
duplicates = df_logit_long[
    df_logit_long.duplicated(
        subset=["participant_id", "timepoint"],
        keep=False
    )
]

print(duplicates)

Empty DataFrame
Columns: [participant_id, en_rand, timepoint, epds_result, timepoint_num, epds_result_num]
Index: []


In [104]:
#checking for available epds data
participant_counts = (
    df_logit_long
    .groupby("participant_id")["epds_result"]
    .nunique()
)

print(participant_counts.value_counts())

epds_result
1    475
2    137
Name: count, dtype: int64


In [105]:
df_logit_long['epds_result'].value_counts(dropna=False)

epds_result
Negative    1014
Positive     117
None          93
Name: count, dtype: int64

In [106]:
summary = (
    df_logit_long
    .groupby(["en_rand", "timepoint"])
    .agg(
        n=("epds_result_num", "size"),
        positive=("epds_result_num", "sum"),
        positivity=("epds_result_num", "mean")
    )
    .reset_index()
)

summary["positivity_percent"] = summary["positivity"] * 100

print(summary)

        en_rand timepoint    n  positive  positivity  positivity_percent
0       Control    2weeks  306      51.0    0.173469           17.346939
1       Control  6_8weeks  306      22.0    0.080882            8.088235
2  Intervention    2weeks  306      33.0    0.113793           11.379310
3  Intervention  6_8weeks  306      11.0    0.040000            4.000000


In [107]:
positivity_table = df_logit_long.pivot_table(
    index="en_rand",
    columns="timepoint",
    values="epds_result_num",
    aggfunc="mean"
) * 100

print(positivity_table)

timepoint        2weeks  6_8weeks
en_rand                          
Control       17.346939  8.088235
Intervention  11.379310  4.000000


In [108]:
positivity_table

timepoint,2weeks,6_8weeks
en_rand,,
Control,17.346939,8.088235
Intervention,11.379310,4.000000


In [109]:
positivity_table["change"] = (
    positivity_table['2weeks']
    - positivity_table['6_8weeks']
)

print(positivity_table)

timepoint        2weeks  6_8weeks    change
en_rand                                    
Control       17.346939  8.088235  9.258703
Intervention  11.379310  4.000000  7.379310


# Logistic GEE

In [110]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [114]:
df_logit_long["group"] = pd.Categorical(
    df_logit_long["en_rand"],
    categories=["Control", "Intervention"]
)

df_logit_long["timepoint"] = pd.Categorical(
    df_logit_long["timepoint"],
    categories=["2weeks", "6_8weeks"]
)

In [120]:
model = smf.gee(
    formula="epds_result_num ~ group * timepoint",
    groups="participant_id",
    data=df_logit_long,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
)

result = model.fit()

print(result.summary())

                               GEE Regression Results                              
Dep. Variable:             epds_result_num   No. Observations:                 1131
Model:                                 GEE   No. clusters:                      590
Method:                        Generalized   Min. cluster size:                   1
                      Estimating Equations   Max. cluster size:                   2
Family:                           Binomial   Mean cluster size:                 1.9
Dependence structure:         Exchangeable   Num. iterations:                     4
Date:                     Wed, 26 Aug 2026   Scale:                           1.000
Covariance type:                    robust   Time:                         10:08:49
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------
Intercept           

In [121]:
params = result.params
conf = result.conf_int()

or_table = pd.DataFrame({
    "OR": np.exp(params),
    "CI_lower": np.exp(conf[0]),
    "CI_upper": np.exp(conf[1]),
    "p_value": result.pvalues
})

print(or_table)

                                                   OR  CI_lower  CI_upper  \
Intercept                                    0.209593  0.154950  0.283504   
group[T.Intervention]                        0.611949  0.381717  0.981045   
timepoint[T.6_8weeks]                        0.413955  0.253209  0.676748   
group[T.Intervention]:timepoint[T.6_8weeks]  0.776630  0.340895  1.769328   

                                                  p_value  
Intercept                                    3.704964e-24  
group[T.Intervention]                        4.140646e-02  
timepoint[T.6_8weeks]                        4.366869e-04  
group[T.Intervention]:timepoint[T.6_8weeks]  5.473522e-01  


In [ ]:
# Get predicted positivity probabilities

prediction_df = pd.DataFrame({
    "group": [
        "Control",
        "Control",
        "Intervention",
        "Intervention"
    ],
    "timepoint": [
        "2weeks",
        "6_8weeks",
        "2weeks",
        "6_8weeks"
    ]
})

prediction_df["predicted_probability"] = (
    result.predict(prediction_df)
)

prediction_df["predicted_percent"] = (
    prediction_df["predicted_probability"] * 100
)

print(prediction_df)


          group timepoint  predicted_probability  predicted_percent
0       Control    2weeks               0.173275          17.327536
1       Control  6_8weeks               0.079835           7.983522
2  Intervention    2weeks               0.113679          11.367937
3  Intervention  6_8weeks               0.039601           3.960132


In [118]:
df_logit_long

,participant_id,en_rand,timepoint,epds_result,timepoint_num,epds_result_num,group
0,PPD-M-001,Control,2weeks,Positive,0,1.0,Control
612,PPD-M-001,Control,6_8weeks,Negative,1,0.0,Control
613,PPD-M-002,Intervention,6_8weeks,Negative,1,0.0,Intervention
1,PPD-M-002,Intervention,2weeks,Negative,0,0.0,Intervention
2,PPD-M-003,Control,2weeks,Negative,0,0.0,Control
...,...,...,...,...,...,...,...
609,PPD-M-610,Control,2weeks,Negative,0,0.0,Control
1222,PPD-M-611,Intervention,6_8weeks,Negative,1,0.0,Intervention
610,PPD-M-611,Intervention,2weeks,Negative,0,0.0,Intervention
611,PPD-M-612,Intervention,2weeks,Negative,0,0.0,Intervention
